# 🧪 PT-W2-D4 概念实验：Rule 抽取——声明式业务规则引擎

> 配套阅读：`PT-W2-D4-Rule抽取.md`
> 三类规则（校验/路由/推理），从自然语言提升为 If-Then 声明。

## 第 1 格：三类 Rule 的 dataclass 定义

In [ ]:
from dataclasses import dataclass
from enum import Enum

class RuleType(Enum):
    VALIDATION = "校验规则"
    ROUTING = "路由规则"
    INFERENCE = "推理规则"

@dataclass
class BusinessRule:
    rule_id: str
    rule_type: RuleType
    when: str
    check: list[str]
    then_pass: str
    then_fail: str
    evidence: str

rules = [
    BusinessRule("RULE-SAL-021", RuleType.VALIDATION, "报价提交时",
                 ["租金>=底价", "租期合法", "保证金>=最低"],
                 "绿色通过", "红色升级审批", "BCM CRE-SAL-021"),
    BusinessRule("RULE-FIN-015", RuleType.ROUTING, "收款核销时",
                 ["账款已核销"],
                 "按优先级核销", "余额转预存款", "BCM CRE-FIN-015"),
    BusinessRule("RULE-CON-024", RuleType.INFERENCE, "合同终止时",
                 ["inspection完成"],
                 "触发 occupancy-effect + financial-effect",
                 "终止受阻：inspection未完成", "BCM CRE-CON-024"),
]
for r in rules:
    print(f"[{r.rule_type.value}] {r.rule_id}: {r.when}")
    print(f"  check: {'; '.join(r.check)}")
    print()

## 第 2 格：声明式规则引擎

In [ ]:
def evaluate_rule(rule, facts):
    unmet = []
    for condition in rule.check:
        if ">=" in condition:
            left, right = condition.split(">=")
            if facts.get(left.strip(), 0) < facts.get(right.strip(), 0):
                unmet.append(condition)
        elif condition not in facts or not facts[condition]:
            unmet.append(condition)
    if not unmet:
        return True, rule.then_pass
    return False, f"{rule.then_fail}: {', '.join(unmet)}"

facts1 = {"租金": 100, "底价": 100, "inspection完成": True}
print("报价判标(达标):", evaluate_rule(rules[0], facts1))
facts2 = {"租金": 80, "底价": 100}
print("报价判标(低价):", evaluate_rule(rules[0], facts2))
facts3 = {"inspection完成": False}
print("合同终止推理:", evaluate_rule(rules[2], facts3))

## 第 3 格：可视化——三类规则分布

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
font_name = font_manager.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc").get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

labels = [t.value for t in RuleType]
sizes = [sum(1 for r in rules if r.rule_type == t) for t in RuleType]
colors = ["#4CAF50", "#2196F3", "#F44336"]
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(labels, sizes, color=colors, edgecolor="white")
ax.set_title("BCM 业务规则分类（示例）")
for bar, s in zip(bars, sizes):
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2, str(s), va="center")
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第10周/d4_rules.png", dpi=100)
plt.show()
print("规则分类图已绘制")

## 第 4 格：推理规则——跨域链路

In [ ]:
print("""推理规则示例（CRE-CON-024 → CRE-LEA-007 → CRE-FIN-008）：
When:  合同终止 (ContractTerminated)
Check: inspection完成?
Then:  occupancy-effect  → 铺位状态 available (→ 03 租赁域)
       financial-effect  → 解约日后应收抹除 (→ 04 财务域)

关键：occupancy 和 financial 是两条独立推理链，
Agent 可以分别追踪"铺位是否释放"和"账单是否停止"。
""")